### Inspect temporal Leiden parameter sweep output

For each `(gamma, omega)` pair in a `community_parameter_sweeps/<run>` folder, cluster the
consensus coassignment matrix (averaged over runs and time windows) to see which ROIs
group together at a given number of communities. Used alongside `summary.csv` to pick
parameters that give the target community count with biologically sensible groupings.

In [4]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform

In [5]:
# Folder containing summary.csv and coassignment_gamma-*_omega-*.csv
SWEEP_DIR = Path("community_parameter_sweeps/sub-ag230908a_20260610_195620")

# Number of communities to try cutting the consensus dendrogram at
K_VALUES = [5, 6, 7]

### Sweep summary

`communities_mean`, `singleton_fraction`, `flexibility_mean` and `coassignment_stability`
for every `(gamma, omega)` pair, sorted by mean community count.

In [6]:
summary = pd.read_csv(SWEEP_DIR / "summary.csv")
summary.sort_values("communities_mean")

,gamma,omega,n_runs,communities_mean,communities_run_sd,communities_min,communities_max,singleton_fraction,flexibility_mean,coassignment_stability,quality_improvement_mean
0,0.25,0.10,20,4.960538,0.017016,2,9,0.066558,0.173758,0.995716,10213.815144
1,0.25,0.25,20,5.018231,0.022815,2,8,0.069846,0.129232,0.994340,12944.017369
2,0.25,0.50,20,5.078385,0.028919,3,8,0.074091,0.097000,0.992049,17621.486476
3,0.30,0.10,20,5.535615,0.019137,3,9,0.085986,0.167898,0.995760,8531.481323
4,0.30,0.25,20,5.619615,0.030972,2,9,0.092712,0.126979,0.994554,11260.160245
5,0.30,0.50,20,5.770000,0.037030,3,9,0.103096,0.091627,0.991898,15962.163975
6,0.35,0.10,20,6.202846,0.019248,3,10,0.114894,0.157897,0.996110,7113.790050
7,0.35,0.25,20,6.355385,0.031083,3,11,0.128447,0.118418,0.994791,9856.736816
8,0.35,0.50,20,6.511692,0.048371,3,10,0.136769,0.083190,0.991780,14590.083970
9,0.40,0.10,20,6.948231,0.023176,3,11,0.152601,0.148132,0.996455,5915.872666


### Expected Networks

1. Salience / anterior DMN — DMNa, SAL Cingulate–insular hub, frequently co-active in rodents.

2. Default Mode Network (posterior) — DMNp, VIS, SUB Retrosplenial/posterior cingulate + visual + subicular — the classic posterior DMN core.

3. Basal ganglia / olfactory-forebrain network — OLF, STR, CTXsp, BF Striatal–olfactory–basal forebrain loop.

4. Auditory–limbic/hippocampal network — AUD, SSs, HCa Auditory cortex grouped with secondary somatosensory and anterior hippocampus.

5. Diencephalic network — TH, HY Thalamus–hypothalamus, a well-documented subcortical relay network.

6. Somatomotor (lateral cortical) network — MOp, SSp Primary motor + primary somatosensory — sensorimotor network.


### Consensus community structure

For each `(gamma, omega)` pair, build a distance matrix from the consensus coassignment
matrix (`1 - coassignment`), run average-linkage hierarchical clustering, and cut the
dendrogram at each value in `K_VALUES` to inspect the resulting ROI groupings.

In [7]:
def consensus_clusters(gamma, omega, k_values):
    df = pd.read_csv(SWEEP_DIR / f"coassignment_gamma-{gamma:g}_omega-{omega:g}.csv", index_col=0)
    rois = df.columns.tolist()

    dist = 1 - df.values
    np.fill_diagonal(dist, 0)
    condensed = squareform((dist + dist.T) / 2, checks=False)
    Z = linkage(condensed, method="average")

    groupings = {}
    for k in k_values:
        clusters = fcluster(Z, k, criterion="maxclust")
        groups = {}
        for roi, c in zip(rois, clusters):
            groups.setdefault(c, []).append(roi)
        groupings[k] = list(groups.values())
    return groupings

In [8]:
for _, row in summary.iterrows():
    gamma, omega = row["gamma"], row["omega"]
    print(f"\n=== gamma={gamma:g} omega={omega:g} (communities_mean={row['communities_mean']:.2f}) ===")
    for k, groups in consensus_clusters(gamma, omega, K_VALUES).items():
        print(f"  k={k}: " + " | ".join(",".join(g) for g in groups))


=== gamma=0.25 omega=0.1 (communities_mean=4.96) ===
  k=5: DMNa,SAL,MOp,SSp | DMNp,AUD,VIS,HCa,SUB | OLF,STR,CTXsp,BF | TH,HY | SSs
  k=6: DMNa,SAL | DMNp,AUD,VIS,HCa,SUB | OLF,STR,CTXsp,BF | TH,HY | MOp,SSp | SSs
  k=7: DMNa,SAL | DMNp,VIS,SUB | OLF,STR,CTXsp,BF | AUD,HCa | TH,HY | MOp,SSp | SSs

=== gamma=0.25 omega=0.25 (communities_mean=5.02) ===
  k=5: DMNa,SAL,MOp,SSp | DMNp,AUD,VIS,HCa,SUB | OLF,STR,CTXsp,BF | TH,HY | SSs
  k=6: DMNa,SAL | DMNp,AUD,VIS,HCa,SUB | OLF,STR,CTXsp,BF | TH,HY | MOp,SSp | SSs
  k=7: DMNa,SAL | DMNp,VIS,SUB | OLF,STR,CTXsp,BF | AUD,HCa | TH,HY | MOp,SSp | SSs

=== gamma=0.25 omega=0.5 (communities_mean=5.08) ===
  k=5: DMNa,SAL,MOp,SSp | DMNp,AUD,VIS,HCa,SUB | OLF,STR,CTXsp,BF | TH,HY | SSs
  k=6: DMNa,SAL | DMNp,AUD,VIS,HCa,SUB | OLF,STR,CTXsp,BF | TH,HY | MOp,SSp | SSs
  k=7: DMNa,SAL | DMNp,VIS,SUB | OLF,STR,CTXsp,BF | AUD,HCa | TH,HY | MOp,SSp | SSs

=== gamma=0.3 omega=0.1 (communities_mean=5.54) ===
  k=5: DMNa,SAL,MOp,SSp | DMNp,AUD,VIS,HCa,SUB